# Senate Money + Polymarket: First-Pass Analysis

This notebook does two things:

1. sanity-checks the current saved Senate finance outputs
2. rebuilds a cleaner **current-cycle** snapshot for descriptive analysis and merges it with Polymarket odds

## Important caveat

The first saved `senate_money_snapshot_2026.csv` mixes in a lot of historical committee totals. That means some rows reflect older election cycles rather than the latest 2025-2026 campaign picture.

This notebook therefore builds a cleaner view directly from `senate_candidate_finance_totals_2026.csv` by keeping rows with `coverage_end_date >= 2025-01-01`, then selecting the top DEM and top REP in each state by `total_receipts`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

processed = ROOT / 'data' / 'processed'
analysis_dir = ROOT / 'analysis'
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')


In [ ]:
snapshot_raw = pd.read_csv(processed / 'senate_money_snapshot_2026.csv', dtype=str)
polymarket = pd.read_csv(processed / 'polymarket_senate_top_two_2026.csv', dtype=str)
totals_raw = pd.read_csv(processed / 'senate_candidate_finance_totals_2026.csv', dtype=str)

print('raw snapshot shape:', snapshot_raw.shape)
print('polymarket shape:', polymarket.shape)
print('raw totals shape:', totals_raw.shape)


## 1. Sanity check the raw saved snapshot

The current saved snapshot is still useful as a recovery artifact, but we should check how stale it is before leaning on it analytically.


In [ ]:
raw_check = snapshot_raw.copy()
for party in ['dem', 'rep']:
    raw_check[f'{party}_coverage_end_date'] = pd.to_datetime(raw_check[f'{party}_coverage_end_date'], errors='coerce')

stale_summary = pd.DataFrame({
    'party': ['DEM', 'REP'],
    'rows_with_date': [raw_check['dem_coverage_end_date'].notna().sum(), raw_check['rep_coverage_end_date'].notna().sum()],
    'pre_2025_rows': [
        (raw_check['dem_coverage_end_date'].dt.year < 2025).sum(),
        (raw_check['rep_coverage_end_date'].dt.year < 2025).sum(),
    ],
})
stale_summary['share_pre_2025'] = stale_summary['pre_2025_rows'] / stale_summary['rows_with_date']
display(stale_summary)

raw_check[['state','dem_candidate_name','dem_coverage_end_date','rep_candidate_name','rep_coverage_end_date']].head(10)


In [ ]:
totals_cycle_check = totals_raw.copy()
totals_cycle_check['coverage_end_date'] = pd.to_datetime(totals_cycle_check['coverage_end_date'], errors='coerce')
cycle_counts = totals_cycle_check['cycle'].fillna('NA').value_counts().rename_axis('cycle').reset_index(name='rows')
coverage_year_counts = totals_cycle_check['coverage_end_date'].dt.year.fillna(-1).astype(int).value_counts().sort_index().rename_axis('coverage_year').reset_index(name='rows')

display(cycle_counts.head(15))
display(coverage_year_counts.tail(12))


## 2. Build a cleaner current-cycle snapshot

Method used here:

- parse the raw committee totals file
- keep rows with `coverage_end_date >= 2025-01-01`
- restrict to DEM and REP
- pick the top fundraiser within each state-party
- pivot to one row per state


In [ ]:
totals = totals_raw.copy()
totals['coverage_end_date'] = pd.to_datetime(totals['coverage_end_date'], errors='coerce')
for col in ['total_receipts', 'total_disbursements', 'cash_on_hand', 'debts_owed_by_committee']:
    totals[f'{col}_num'] = pd.to_numeric(totals[col], errors='coerce').fillna(0)

def normalize_party(value):
    value = str(value).strip().upper()
    mapping = {
        'DEM': 'DEM', 'D': 'DEM', 'DEMOCRAT': 'DEM', 'DEMOCRATIC': 'DEM',
        'REP': 'REP', 'R': 'REP', 'REPUBLICAN': 'REP'
    }
    return mapping.get(value, 'OTHER')

totals['party_normalized'] = totals['party'].apply(normalize_party)
current = totals[(totals['coverage_end_date'] >= '2025-01-01') & (totals['party_normalized'].isin(['DEM', 'REP']))].copy()
current_top = (
    current.sort_values(['state', 'party_normalized', 'total_receipts_num'], ascending=[True, True, False])
    .groupby(['state', 'party_normalized'], as_index=False)
    .head(1)
    .copy()
)

current_top[['state', 'candidate_name', 'party_normalized', 'total_receipts', 'coverage_end_date']].head(12)


In [ ]:
current_wide = current_top.pivot(
    index='state',
    columns='party_normalized',
    values=[
        'candidate_name',
        'fec_candidate_id',
        'committee_id',
        'total_receipts_num',
        'total_disbursements_num',
        'cash_on_hand_num',
        'debts_owed_by_committee_num',
        'coverage_end_date',
    ],
)
current_wide.columns = [f'{a.lower()}_{b.lower()}' for a, b in current_wide.columns]
current_wide = current_wide.reset_index()
print('current-cycle snapshot rows:', len(current_wide))
display(current_wide.head())


## 3. Merge with Polymarket

For party-level markets we infer the favorite side from the market label (`Democrat` / `Republican`).
For candidate-level markets like Alaska, we infer the favorite side by matching the Polymarket candidate FEC ID to the current-cycle DEM/REP candidates.


In [ ]:
odds = polymarket.copy()
for col in ['top_1_yes_probability', 'top_2_yes_probability']:
    odds[col] = pd.to_numeric(odds[col], errors='coerce')

merged = current_wide.merge(odds, on='state', how='left')

def infer_favorite_party(row):
    entity_type = row.get('top_1_entity_type')
    entity_name = str(row.get('top_1_entity_name') or '').strip().upper()
    if entity_type == 'party':
        if entity_name == 'DEMOCRAT':
            return 'DEM'
        if entity_name == 'REPUBLICAN':
            return 'REP'
        return None
    top_id = row.get('top_1_fec_candidate_id')
    if pd.notna(top_id) and top_id == row.get('fec_candidate_id_dem'):
        return 'DEM'
    if pd.notna(top_id) and top_id == row.get('fec_candidate_id_rep'):
        return 'REP'
    return None

merged['favorite_party'] = merged.apply(infer_favorite_party, axis=1)
merged['favorite_margin_prob'] = merged['top_1_yes_probability'] - merged['top_2_yes_probability']
merged['combined_receipts'] = merged['total_receipts_num_dem'].fillna(0) + merged['total_receipts_num_rep'].fillna(0)
merged['combined_spend'] = merged['total_disbursements_num_dem'].fillna(0) + merged['total_disbursements_num_rep'].fillna(0)
merged['combined_cash'] = merged['cash_on_hand_num_dem'].fillna(0) + merged['cash_on_hand_num_rep'].fillna(0)
merged['dem_receipts_share'] = merged['total_receipts_num_dem'] / merged['combined_receipts']
merged['dem_cash_share'] = merged['cash_on_hand_num_dem'] / merged['combined_cash']
merged['favorite_receipts_share'] = merged.apply(
    lambda r: r['dem_receipts_share'] if r['favorite_party'] == 'DEM' else (1 - r['dem_receipts_share'] if r['favorite_party'] == 'REP' else np.nan),
    axis=1,
)
merged['favorite_cash_share'] = merged.apply(
    lambda r: r['dem_cash_share'] if r['favorite_party'] == 'DEM' else (1 - r['dem_cash_share'] if r['favorite_party'] == 'REP' else np.nan),
    axis=1,
)
merged['receipts_gap_share'] = (merged['total_receipts_num_dem'] - merged['total_receipts_num_rep']).abs() / merged['combined_receipts']
merged['cash_gap_share'] = (merged['cash_on_hand_num_dem'] - merged['cash_on_hand_num_rep']).abs() / merged['combined_cash']

print('states in current-cycle snapshot:', len(merged))
print('states with Polymarket odds:', merged['top_1_yes_probability'].notna().sum())
print('states with inferred favorite party:', merged['favorite_party'].notna().sum())


## 4. Quick findings from the cleaned cut

The point here is not to make a strong causal claim. We are just checking whether the basic descriptive patterns line up with intuition.


In [ ]:
def corr_table(df, pairs):
    rows = []
    for left, right in pairs:
        sub = df[[left, right]].dropna()
        rows.append({
            'left_metric': left,
            'right_metric': right,
            'correlation': sub[left].corr(sub[right]),
            'n': len(sub),
        })
    return pd.DataFrame(rows)

corrs = corr_table(
    merged,
    [
        ('favorite_margin_prob', 'combined_receipts'),
        ('favorite_margin_prob', 'combined_spend'),
        ('favorite_margin_prob', 'combined_cash'),
        ('favorite_margin_prob', 'favorite_receipts_share'),
        ('favorite_margin_prob', 'favorite_cash_share'),
        ('top_1_yes_probability', 'favorite_receipts_share'),
        ('top_1_yes_probability', 'favorite_cash_share'),
    ],
)
display(corrs.sort_values('correlation', ascending=False))


In [ ]:
closest_races = merged.sort_values('favorite_margin_prob')[[
    'state',
    'candidate_name_dem',
    'candidate_name_rep',
    'top_1_entity_name',
    'top_1_yes_probability',
    'top_2_entity_name',
    'top_2_yes_probability',
    'combined_receipts',
    'combined_cash',
    'receipts_gap_share',
]]

display(closest_races.head(12))


In [ ]:
most_expensive = merged.sort_values('combined_receipts', ascending=False)[[
    'state',
    'candidate_name_dem',
    'candidate_name_rep',
    'combined_receipts',
    'combined_spend',
    'combined_cash',
    'favorite_margin_prob',
]]

display(most_expensive.head(12))


In [ ]:
money_market_tension = merged[(merged['favorite_receipts_share'].notna()) & (merged['favorite_receipts_share'] < 0.5)][[
    'state',
    'favorite_party',
    'top_1_yes_probability',
    'favorite_margin_prob',
    'favorite_receipts_share',
    'favorite_cash_share',
    'candidate_name_dem',
    'candidate_name_rep',
]].sort_values('top_1_yes_probability', ascending=False)

display(money_market_tension)


## 5. Charts

Two simple visuals:

- does the market favorite usually also hold the larger share of receipts?
- are tighter races attracting more total money?


In [ ]:
plot_df = merged.dropna(subset=['top_1_yes_probability', 'favorite_receipts_share', 'combined_receipts']).copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(plot_df['favorite_receipts_share'], plot_df['top_1_yes_probability'], alpha=0.8)
axes[0].axvline(0.5, color='gray', linestyle='--', linewidth=1)
axes[0].set_xlabel('Favorite share of combined receipts')
axes[0].set_ylabel('Favorite Polymarket win probability')
axes[0].set_title('Favorite Money Share vs Favorite Win Probability')

axes[1].scatter(plot_df['favorite_margin_prob'], plot_df['combined_receipts'], alpha=0.8)
axes[1].set_yscale('log')
axes[1].set_xlabel('Favorite probability margin')
axes[1].set_ylabel('Combined receipts (log scale)')
axes[1].set_title('Race Tightness vs Combined Receipts')

plt.tight_layout()
plt.show()


In [ ]:
bar_df = merged.dropna(subset=['favorite_margin_prob']).sort_values('favorite_margin_prob').head(10).copy()
bar_df['label'] = bar_df['state'] + ' | ' + bar_df['top_1_entity_name']

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(bar_df['label'], bar_df['favorite_margin_prob'])
ax.set_xlabel('Favorite probability margin')
ax.set_title('Closest Polymarket Senate Races in the Clean Current-Cycle Cut')
plt.tight_layout()
plt.show()


## 6. Save a derived analysis table

This gives you one merged flat file to use outside the notebook if you want to explore in Sheets, pandas, or a BI tool.


In [ ]:
derived_path = analysis_dir / 'senate_money_polymarket_current_cycle_analysis.csv'
merged.to_csv(derived_path, index=False)
print('saved:', derived_path)


## Preliminary read from this notebook build

From the cleaned current-cycle cut I saw a few useful first-pass patterns:

- the **favorite's share of money** is moderately positively associated with Polymarket win probability
- the **favorite's share of money** is more informative than simple total dollars alone
- **combined spending/receipts are not strongly higher in the safest races**; if anything the sign is slightly negative, which fits the idea that competitive races attract money
- several races look interesting because the market favorite is only modestly ahead despite a noticeable fundraising edge or deficit

This is all descriptive, not causal, but it is enough for a useful v1 monitoring notebook.
